# Domain 2: Tool Design & MCP Integration
## Task Statement 2.5: Select and Apply Built-in Tools Effectively
**Certification Weight**: 20% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand when and how Claude Code selects and applies built-in developer tools (`Read`, `Write`, `Edit`, `Bash`, `Grep`, `Glob`) and how to handle tool failure recovery.

---

## 🏗️ Built-in Tools Decision Matrix

| Tool | Primary Purpose | When to Choose | When to Avoid |
|---|---|---|---|
| **`Grep`** | Content pattern search | Finding function definitions, symbol usages, or error strings across files | Searching for file names or directory structures |
| **`Glob`** | File path matching | Locating files by filename pattern (e.g. `**/*.py`, `tests/test_*.py`) | Searching within file contents |
| **`Read`** | Full or chunked file inspection | Reading context around a known line range or inspecting complete files | Checking if a keyword exists across 50 files (use `Grep` first) |
| **`Edit`** | Targeted surgical replacement | Modifying specific functions or lines while preserving existing formatting | Creating brand new files; replacing entire multi-hundred line files |
| **`Write`** | Full file creation or overwrite | Creating new files from scratch, or complete rewrites when `Edit` fails | Small targeted bug fixes in existing large files |
| **`Bash`** | Shell execution | Running linters, test suites, git commands, and build tools | Simple file operations that built-in tools handle safer |

---

## ⚡ The `Edit` Fallback Protocol

On the exam, scenarios often ask what happens when `Edit` fails:
```mermaid
flowchart TD
    A["Attempt Edit(old_str, new_str)"] --> B{"Edit Status?"}
    B -->|"Success"| C["Code Updated"]
    B -->|"Multiple matches found"| D["Read context around target line"]
    B -->|"Target text not found"| D
    D --> E["Expand anchor string<br>to include unique lines"]
    E --> F["Re-attempt Edit with unique context"]
    F -->|"Still failing"| G["Fall back to Read + Write<br>(full file replacement)"]
```

### Key Gotcha
- If `old_str` matches 2+ locations in the target file, `Edit` **fails** to prevent unintended replacements.
- **Resolution**: Expand the anchor lines above/below to ensure uniqueness, or fallback to `Read` followed by clean `Write`.

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import os
import re
import sys
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.5 Built-in Tools Usage.")

### 🧰 Step 2: Built-in Toolbox Engine Implementation
We implement the full suite of built-in developer tools with safe fallback handling.

In [ ]:
# Step 2: BuiltinToolbox Implementation
class BuiltinToolbox:
    def __init__(self, workspace_root: Path) -> None:
        self.root = workspace_root

    def glob_files(self, pattern: str) -> List[str]:
        matches = []
        ignore_dirs = {".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache"}
        for dirpath, dirnames, filenames in os.walk(str(self.root)):
            dirnames[:] = [d for d in dirnames if d not in ignore_dirs and not d.startswith(".venv")]
            rel_dir = os.path.relpath(dirpath, str(self.root)).replace("\\", "/")
            for fname in filenames:
                rel = f"{rel_dir}/{fname}" if rel_dir != "." else fname
                if fnmatch.fnmatch(rel, pattern) or fnmatch.fnmatch(fname, pattern):
                    matches.append(rel)
        return sorted(matches)

    def grep_search(self, regex_pattern: str, file_pattern: str = "*") -> List[Dict[str, Any]]:
        results = []
        compiled = re.compile(regex_pattern)
        for rel_path in self.glob_files(file_pattern):
            full_path = self.root / rel_path
            try:
                lines = full_path.read_text(encoding="utf-8").splitlines()
                for line_no, line in enumerate(lines, start=1):
                    if compiled.search(line):
                        results.append({"file": rel_path, "line": line_no, "content": line.strip()})
            except Exception:
                continue
        return results

    def read_file(self, rel_path: str, offset: int = 1, limit: Optional[int] = None) -> Dict[str, Any]:
        full_path = self.root / rel_path
        if not full_path.exists():
            return {"success": False, "error": f"File not found: {rel_path}"}
        lines = full_path.read_text(encoding="utf-8").splitlines()
        start_idx = max(0, offset - 1)
        end_idx = len(lines) if limit is None else start_idx + limit
        slice_lines = lines[start_idx:end_idx]
        return {
            "success": True,
            "file": rel_path,
            "start_line": offset,
            "total_lines": len(lines),
            "content": "\n".join(slice_lines),
        }

    def write_file(self, rel_path: str, content: str) -> Dict[str, Any]:
        full_path = self.root / rel_path
        full_path.parent.mkdir(parents=True, exist_ok=True)
        full_path.write_text(content, encoding="utf-8")
        return {"success": True, "file": rel_path}

    def edit_file(self, rel_path: str, old_str: str, new_str: str) -> Dict[str, Any]:
        read_res = self.read_file(rel_path)
        if not read_res["success"]:
            return read_res
        content = read_res["content"]
        count = content.count(old_str)
        if count == 0:
            return {"success": False, "error": "Anchor not found."}
        if count > 1:
            return {"success": False, "error": f"Ambiguous edit: found {count} occurrences."}
        updated = content.replace(old_str, new_str, 1)
        self.write_file(rel_path, updated)
        return {"success": True, "file": rel_path}

    def execute_edit_with_fallback(self, rel_path: str, old_str: str, new_str: str, unique_context: str) -> Dict[str, Any]:
        res = self.edit_file(rel_path, old_str, new_str)
        if not res["success"]:
            # Fallback: expand anchor with surrounding unique context
            expanded_old = unique_context
            expanded_new = unique_context.replace(old_str, new_str)
            res_retry = self.edit_file(rel_path, expanded_old, expanded_new)
            if res_retry["success"]:
                return {"status": "EDIT_RETRY_SUCCESS", "details": res_retry}
            # Ultimate fallback: read full file, replace, and write
            read_full = self.read_file(rel_path)
            updated = read_full["content"].replace(old_str, new_str, 1)
            self.write_file(rel_path, updated)
            return {"status": "READ_WRITE_FALLBACK_SUCCESS"}
        return {"status": "DIRECT_EDIT_SUCCESS"}

print(" BuiltinToolbox ready.")

### 🧪 Step 3: Interactive Simulation
We create a temporary workspace, write a file with ambiguous targets, trigger the ambiguity failure, and succeed via fallback.

In [ ]:
# Step 3: Simulation
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    toolbox = BuiltinToolbox(tmp_path)

    # 1. Write file with repeated text
    sample_content = "def func_a(): return 1\ndef func_b(): return 1\n"
    toolbox.write_file("app.py", sample_content)

    # 2. Attempt ambiguous edit
    amb_res = toolbox.edit_file("app.py", "return 1", "return 2")
    print("Ambiguous Edit Attempt:", amb_res)

    # 3. Execute edit with fallback (expanding context to 'def func_a(): return 1')
    fallback_res = toolbox.execute_edit_with_fallback(
        "app.py",
        old_str="return 1",
        new_str="return 2",
        unique_context="def func_a(): return 1"
    )
    print("\nFallback Edit Execution:", fallback_res)

    # 4. Read updated content
    final_content = toolbox.read_file("app.py")
    print("\nUpdated File Content:\n", final_content["content"])

### 🔬 Step 4: Verification Assertions
We assert that:
1. Ambiguous edit fails with clear error.
2. Context expansion fallback replaces only the targeted function.
3. `func_a` returns 2, and `func_b` remains returning 1.

In [ ]:
# Step 4: Verification Checks
assert amb_res["success"] is False
assert "Ambiguous edit" in amb_res["error"]
assert fallback_res["status"] == "EDIT_RETRY_SUCCESS"
assert "def func_a(): return 2" in final_content["content"]
assert "def func_b(): return 1" in final_content["content"]

print(" All verification assertions passed! Built-in tools verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Follow the 3-step codebase discovery workflow: **Glob -> Grep -> Read (slices)**.
- **Rule 2**: Avoid dumping whole files into context when only specific functions are needed.
- **Rule 3**: When an edit fails due to ambiguity, expand the unique context lines or fall back to Read + Write.